# Evo 1.3 — FCUSD data-first training
This notebook refuses to train without real PeopleSense samples, measured drill outcomes, and confirmed Pi coordinates. It does not promote a model unless every quality gate passes.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json
drive.mount('/content/drive')
REPO_DIR = Path('/content/drive/MyDrive/Evo')  # change if your Drive folder differs
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/rvkshn-10/Evo', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(REPO_DIR)

In [ ]:
subprocess.run(['pip', 'install', '-q', '-r', 'model_training/evo1_3/requirements-colab.txt'], check=True)

## Required inputs
Upload real PeopleSense XML/JSON files to `data/incoming/evo1.3/peoplesense/` and measured outcomes to `data/incoming/evo1.3/real_outcomes.json`. Set `COORDS_CONFIRMED=True` only after FCUSD verifies all three Pi GPS submissions.

In [ ]:
COORDS_CONFIRMED = False
PEOPLESENSE_DIR = Path('data/incoming/evo1.3/peoplesense')
OUTCOMES = Path('data/incoming/evo1.3/real_outcomes.json')
preflight = ['python', 'model_training/evo1_3/train_evo1_3.py', '--peoplesense-dir', str(PEOPLESENSE_DIR), '--real-outcomes', str(OUTCOMES), '--preflight-only']
if COORDS_CONFIRMED:
    preflight.append('--coords-confirmed')
result = subprocess.run(preflight, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Evo 1.3 preflight blocked; do not train'

In [ ]:
train = ['python', 'model_training/evo1_3/train_evo1_3.py', '--peoplesense-dir', str(PEOPLESENSE_DIR), '--real-outcomes', str(OUTCOMES), '--coords-confirmed', '--output-dir', 'artifacts/evo1.3']
subprocess.run(train, check=True)

In [ ]:
report = json.loads(Path('artifacts/evo1.3/validation_report.json').read_text())
print(json.dumps({k: report[k] for k in ('all_quality_gates_pass', 'failed_quality_gates', 'failure_classification', 'promotion_recommendation')}, indent=2))
assert report['all_quality_gates_pass'], 'Keep Evo 1.2 hybrid; Evo 1.3 is not promotable'